# 07 — Ranking and comparison-table assembly

This notebook confirms how the benchmark turns per-architecture metric records into a ranked leaderboard and comparison tables. We drive the real `ComparisonReport` writer on mock `TrialRecord`s, then read back the generated markdown and reconstruct the leaderboard as a plot.

Modules exercised: `pipelines/benchmark_pipeline/results.py` (`ComparisonReport`, `_HEADLINE_METRICS`, the `_leaderboard`/`_direction` ranking logic invoked through `write_all`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(False)

DEVICE = torch.device("cpu")

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.axisbelow" : True,
})

print("bootstrap ready  —  torch", torch.__version__, " device", DEVICE)


## Mock records with a known correct ranking

We construct records whose metrics improve uniformly from worst to best model, so the correct leaderboard order is known in advance. This lets us verify the real ranking logic by eye.

In [ ]:
import tempfile
from pathlib import Path

from pipelines.benchmark_pipeline.results import ComparisonReport, TrialRecord, _HEADLINE_METRICS
from tools.logger import Logger

models       = ["alpha", "bravo", "charlie", "delta"]
HIGHER_KEYS  = {"overall_r2_gt", "psnr_db_gt", "pixel_r2_gt_mean", "pixel_cosine_gt_mean", "ssim_gt_elev_mean"}

def metrics_for(rank_index, n):
    skill = 1.0 - rank_index / (n - 1)
    out   = {}
    for key, _ in _HEADLINE_METRICS:
        out[key] = 0.5 + 0.4 * skill if key in HIGHER_KEYS else 1.0 - 0.6 * skill
    return out

records = []
for i, name in enumerate(models):
    record            = TrialRecord(name=name, run_dir=Path("/tmp") / name)
    record.parameters = 300_000
    record.size_match = {"scale": 1.0, "deviation_pct": 0.0, "overrides": {}}
    record.overfit    = {"status": "PASS", "final_loss": 1e-4, "converged": True}
    record.training_result = {"status": "DONE", "duration_s": 120.0}
    record.checkpoint = {"best_epoch": 10, "best_val_loss": 0.4, "n_train_epochs": 20}
    record.metrics    = metrics_for(i, len(models))
    records.append(record)

print("expected best-to-worst order:", models)

## Running the real comparison writer

`ComparisonReport.write_all` produces the overview (with the leaderboard), the metrics comparison, the figure/gif galleries (empty here), and a summary JSON. We write into a temporary directory.

In [ ]:
out_dir = Path(tempfile.mkdtemp()) / "comparison"
logger  = Logger(log_dir=str(out_dir.parent), name="comparison_demo")

report  = ComparisonReport(
    records         = records,
    out_dir         = out_dir,
    reference_model = "alpha",
    embed_images    = False,
    logger          = logger,
)
written = report.write_all()
logger.close()

for path in written:
    print(path.name)

## Reading back the leaderboard

We extract the leaderboard section from the generated overview markdown and print it verbatim. This is the genuine output of the ranking logic, not a reimplementation.

In [ ]:
overview = (out_dir / "benchmark_overview.md").read_text(encoding="utf-8")
section  = overview[overview.index("## Leaderboard"):]
print(section)

## Parsing mean ranks for a plot

We parse the leaderboard table rows to recover each model's mean rank, then plot them. A correct ranking shows the models in the order we constructed, with mean rank increasing from the best model to the worst.

In [ ]:
import re

rows = re.findall(r"\|\s*\d+\s*\|\s*`([^`]+)`\s*\|\s*([\d.]+)\s*\|", section)
rank_names = [name for name, _ in rows]
mean_ranks = [float(value) for _, value in rows]

for name, mr in zip(rank_names, mean_ranks):
    print(f"{name:10s} mean rank {mr:.2f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(rank_names, mean_ranks, color="#3b6ea5")
ax.set_ylabel("mean rank across headline metrics (lower = better)")
ax.set_title("Reconstructed leaderboard from ComparisonReport output")
fig.tight_layout()
plt.show()

assert rank_names == models, "leaderboard order did not match the constructed skill order"
print("\nleaderboard order matches the constructed best-to-worst order")

## Expected visual outcome

The printed leaderboard lists the four models in the order alpha, bravo, charlie, delta, with mean rank rising 1, 2, 3, 4. The bar chart reproduces that monotone increase and the final assertion passes, confirming the real ranking logic orders models by metric skill as intended.